In [1]:
import re

In [6]:
re.search("tricks", "/Users/alifouladgar/data_structure_algorithm/techniques/python-tricks/Python_Tricks_Notebook.ipynb")

<re.Match object; span=(63, 69), match='tricks'>

In [7]:
# re methods
re.findall("tricks", "/Users/alifouladgar/data_structure_algorithm/techniques/python-tricks/Python_Tricks_Notebook.ipynb")

['tricks']

In [8]:
re.split("tricks", "/Users/alifouladgar/data_structure_algorithm/techniques/python-tricks/Python_Tricks_Notebook.ipynb")

['/Users/alifouladgar/data_structure_algorithm/techniques/python-',
 '/Python_Tricks_Notebook.ipynb']

In [16]:
pattern = r"\d{3}:\d{2}:\d{3}"
text = "Here is the number 234:89:777"
match = re.search(pattern, text)
print(match.group)


<built-in method group of re.Match object at 0x107ce7a40>


In [17]:
match = re.findall(pattern, text)
match

['234:89:777']

In [22]:
text = "ali@fouladgar.com,jane@smith.com"
pattern = r"@"
re.split(pattern, text)

['ali', 'fouladgar.com,jane', 'smith.com']

In [44]:
from collections import Counter
text = "Here is the number 234:89:777"
pattern = r"[\w]+"
Counter(re.findall(pattern, text.lower())).most_common(2)

[('here', 1), ('is', 1)]

In [55]:
text = "Here is the number 234:89:777"
pattern = r"[^0-9A-Za-z]+"
re.findall(pattern, text.lower())

[' ', ' ', ' ', ' ', ':', ':']

In [51]:
text = "Here is the number 234:89:777"
pattern = r"\w+"
for match in re.finditer(pattern,text):
    print(f"{match.group()} found at index: {match.start()}" )

Here found at index: 0
is found at index: 5
the found at index: 8
number found at index: 12
234 found at index: 19
89 found at index: 23
777 found at index: 26


In [61]:
text = "Here is the number 234:89:777"
pattern = r"[^0-9A-Za-z]+"
match1 = re.sub(pattern, "", text.lower())
pattern2 = r"[\d]+"
match1 = re.sub(pattern2, "", match1)
match1

'hereisthenumber'

In [ ]:
import argparse
import fnmatch
import os
import re
import sys
from pathlib import Path


def parse_args():
    parser = argparse.ArgumentParser(
        description="Search for a regex pattern in files."
    )

    parser.add_argument("pattern")
    parser.add_argument("paths", nargs="+")

    parser.add_argument(
        "-n",
        "--line-number",
        action="store_true",
        help="Show the line number for each match",
    )

    parser.add_argument(
        "-i",
        "--ignore-case",
        action="store_true",
        help="Search case-insensitively",
    )

    parser.add_argument(
        "-l",
        "--files-with-matches",
        action="store_true",
        help="Print only the filename once if the file contains at least one match",
    )

    parser.add_argument(
        "--exclude-dir",
        action="append",
        default=[],
        help="Skip excluded directory names. May be passed multiple times.",
    )

    parser.add_argument(
        "--include",
        action="append",
        default=[],
        help="Only search files whose basename matches this glob. May be passed multiple times.",
    )

    return parser


def should_include_file(path, include_globs):
    if not include_globs:
        return True

    basename = Path(path).name
    return any(fnmatch.fnmatch(basename, glob) for glob in include_globs)


def iter_paths(raw_paths, exclude_dirs, include_globs):
    for raw_path in raw_paths:
        path = Path(raw_path)

        if path.is_file():
            if should_include_file(path, include_globs):
                yield path

        elif path.is_dir():
            for root, dirnames, filenames in os.walk(path):
                dirnames[:] = [
                    dirname for dirname in dirnames
                    if dirname not in exclude_dirs
                ]

                for filename in filenames:
                    file_path = Path(root) / filename

                    if file_path.is_file() and should_include_file(file_path, include_globs):
                        yield file_path

        else:
            print(f"psearch: {path}: no such file or directory", file=sys.stderr)


def search_file(path, regex, show_filename, show_line_number, files_with_matches):
    found = False

    try:
        with open(path, mode="r", encoding="utf-8", errors="replace") as f:
            for line_number, line in enumerate(f, start=1):
                if regex.search(line):
                    found = True

                    if files_with_matches:
                        print(path)
                        return True

                    prefix_parts = []

                    if show_filename:
                        prefix_parts.append(str(path))

                    if show_line_number:
                        prefix_parts.append(str(line_number))

                    if prefix_parts:
                        prefix = ":".join(prefix_parts)
                        print(f"{prefix}:{line}", end="")
                    else:
                        print(line, end="")

    except OSError as err:
        print(f"psearch: {path}: {err}", file=sys.stderr)

    return found


def main(argv=None):
    parser = parse_args()
    args = parser.parse_args(argv)

    regex_flags = re.IGNORECASE if args.ignore_case else 0

    try:
        regex = re.compile(args.pattern, regex_flags)
    except re.error as err:
        print(f"psearch: invalid regex: {err}", file=sys.stderr)
        return 2

    exclude_dirs = set(args.exclude_dir)
    include_globs = args.include

    show_filename = (
        len(args.paths) > 1
        or any(Path(raw_path).is_dir() for raw_path in args.paths)
    )

    any_match = False

    for path in iter_paths(args.paths, exclude_dirs, include_globs):
        matched = search_file(
            path=path,
            regex=regex,
            show_filename=show_filename,
            show_line_number=args.line_number,
            files_with_matches=args.files_with_matches,
        )
        any_match = any_match or matched

    return 0 if any_match else 1


if __name__ == "__main__":
    raise SystemExit(main())

In [64]:
pattern = r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\s+(INFO|WARN|ERROR|DEBUG)\b"
text = "2026-05-05 10:01:22 INFO service started"
re.findall(pattern, text)

[('2026-05-05 10:01:22', 'INFO')]

In [65]:
import re

pattern = r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\s+(INFO|WARN|ERROR|DEBUG)\b"

line = "2026-05-05 10:02:11 ERROR database timeout"

match = re.search(pattern, line)

if match:
    print(match.group(1))  # timestamp
    print(match.group(2))  # log level

2026-05-05 10:02:11
ERROR
